<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Day 5 — The Atlas capstone

<small>Data Creators › Day 5 — anchor notebook</small>

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>

<a id="how-to-use"></a>

## About this notebook

- Day 5's anchor: the agenda, then one numbered section per topic. Each section runs as a cycle: a demo, an activity or challenge, a shareback, a break.
- Every concept has its diagram; every demo has the cells that run it; every activity has its brief; every challenge points into the training repository.
- Run the **Setup** cell first. Cells that start with `!` run in a shell in this notebook's folder with the demo environment on `PATH`.
- 🖊️ Whiteboard lines are the questions the room works through together.
- The last section, **Questions from the room**, collects the day's questions and their answers as they come.

In [ ]:
# Setup: the demo environment on PATH, DuckDB and pandas for showing results, a quick diagram helper
import os, sys, subprocess, textwrap
from pathlib import Path
from IPython.display import Image, Markdown, display

VENV = Path(os.environ.get("CG_DXP_VENV", Path.home() / ".venvs" / "cg-dxp"))
os.environ["PATH"] = ":".join([str(VENV / "bin"), str(Path.home() / ".local" / "bin"),
                               "/Applications/Docker.app/Contents/Resources/bin", "/opt/homebrew/bin", os.environ["PATH"]])
os.environ.setdefault("DBT_SEND_ANONYMOUS_USAGE_STATS", "false")
HERE = Path.cwd()                      # courseware/day5-*/
COMMON = HERE.parent / "common"
assert COMMON.exists(), "Open this notebook from its own folder (courseware/day5-*)"

def quick_diagram(mermaid: str, name: str = "scratch"):
    """Render a Mermaid diagram typed live and show it inline (needs node; first use downloads mermaid-cli)."""
    src = HERE / f"{name}.mmd"; out = HERE / f"{name}.png"
    src.write_text(textwrap.dedent(mermaid).strip() + "\n")
    cfg = COMMON / "diagrams" / "mermaid-config.json"
    r = subprocess.run(["npx", "-y", "-p", "@mermaid-js/mermaid-cli@11.17.0", "mmdc", "-i", str(src), "-o", str(out), "-c", str(cfg), "-b", "white", "-s", "2"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:]); return
    display(Image(filename=str(out)))

def sql(query: str, db: Path, read_only: bool = True):
    """Run a query against a DuckDB file and show it as a table."""
    import duckdb, pandas as pd
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
    con = duckdb.connect(str(db), read_only=read_only)
    try:
        return con.execute(query).df()
    finally:
        con.close()

print("✅ Day 5 setup:", subprocess.run(["dbt", "--version"], capture_output=True, text=True).stdout.split("\n")[1].strip() if subprocess.run(["which", "dbt"], capture_output=True).returncode == 0 else "dbt not on PATH: run common/env/setup-env.sh")

## Today

The day ends when every table has completed Phase 1 and presented. Cohorts usually present early afternoon.

| # | Section | What happens |
|---|---|---|
| 1 | **[The brief and discovery](#s1)** | Brief: the day, the rules, ten minutes of reading · Activity 12: discovery worksheet · Playback: two minutes per table |
| 2 | **[Phase 1: the baseline](#s2)** | Challenge: Atlas Phase 1 · Circulating: section notes, expected results, blockers · Lunch (flexible) |
| 3 | **[Phase 2: extensions and presentation prep](#s3)** | Challenge: the fourteen extensions · Presentation prep: technical work stops at 14:30 |
| 4 | **[Presentations and close](#s4)** | Activity 13: presentations · Tally sheet and certification evidence · Close |

<a id="s1"></a>

## 1. The brief and discovery

### Brief — the day, the rules, ten minutes of reading

![Friday in one line](../common/diagrams/png/d5-capstone-day.png)

- Discovery first, then the baseline, then extensions, then presentations.
- Phase 1 is the certification bar: every team establishes it; Phase 2 is for teams that have and can explain it.

### Activity 12 — discovery worksheet, then playback

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Seven CSV files and seven output tables. Where do you start?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>From the consumer's question backwards to the sources: the discovery worksheet.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Teams that map first build faster and present better; teams that start with SQL find the definitions mid-afternoon.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Roles, two outputs, the four worksheet sections, a two-minute playback.</div></div></div>

![The Atlas sources: one fact table and six dimensions](../common/diagrams/png/d5-atlas-erd.png)

- Trades carry identifiers only; follow portfolio_id to the client and advisor, instrument_id to the security.
- Prices are in cents, fees in basis points, timestamps in epoch seconds: three unit conversions to agree before any SQL.
- No FX conversion in this dataset: money stays in the trade currency, a known simplification to write down.

![What Phase 1 builds: seeds, source-aligned, domain-aligned, consumer-aligned](../common/diagrams/png/d5-atlas-pipeline.png)

- src_ models are faithful pass-throughs; dom_ models conform one definition per entity; rpt_, agg_, ts_ and api_ are shaped for a consumer.
- dom_trading__trade (valid trades, signed quantity, gross value, fee) and dom_trading__latest_price feed almost everything.
- The four reconciliation tests tie the marts together: advisor books, segments and regions must sum to the portfolio summary; three marts must agree on trade counts.

![The names: src_, dom_, rpt_/agg_/ts_/api_, and the older layer words you will hear](../common/diagrams/png/d5-alignment-naming.png)

- Alignment says whose shape a table is built for; medallion and raw/curated names say how far it has been cleaned.
- Communicate in alignment language; recognise the older names when you meet them.

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 12 &nbsp;—&nbsp; Atlas discovery worksheet &nbsp;·&nbsp; Tables &nbsp;·&nbsp; 60 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Before anyone builds, understand the product. Choose at least two business-facing outputs (for example `rpt_portfolio_summary`, `rpt_advisor_book`, `agg_segment_aum`, `agg_instrument_activity`, `agg_fund_nav`, `ts_daily_market`, `api_latest_price`) and work backwards from consumer need to source data. Assign roles: facilitator, scribe, quality challenger, playback spokesperson. Fill one shared worksheet with the four sections below. Describe transformations in plain English; no SQL, no physical keys, no schema design yet.</strong><br><br><strong>Produce:</strong><br>• Producer–creator–consumer map: who supplies, who creates, who consumes each chosen output<br>• Consumer-backwards map: consumer need | output | required sources | rules and assumptions<br>• Mini-STTM for four to six important fields: target field | business meaning | source field(s) | transformation in English | check<br>• Shared glossary (AUM, position, latest price, gross value, fee, basis point, advisor book, valid trade) and trust questions (latest-price selection, invalid trades, fee calculation, reconciliation, no FX conversion)<br><br><strong>Finished early?</strong> Draft a PRD or a data contract with Catalyst for one output.<br><br><strong>Debrief questions:</strong><br>• Which definition did the table disagree about, and how did you settle it?<br>• Which assumption could make the result untrustworthy?<br>• Who is the consumer, and what decision changes?</div></div>

<details><summary><strong>Hints for circulating during discovery</strong></summary>

- **rpt_portfolio_summary** needs positions (trades netted per portfolio and instrument), the latest price per instrument, and the portfolio, client and advisor dimensions. AUM is net quantity times latest price; fees are gross value times bps over 10,000.
- **rpt_advisor_book** is the summary rolled up to advisor; **agg_segment_aum** rolled up to client segment; both must sum to the summary.
- **agg_instrument_activity** and **ts_daily_market** count trades; so does the position mart; the three must agree.
- **agg_fund_nav** weights the latest price by fund holdings; **api_latest_price** is a thin lookup over the latest price.
- Trust questions rooms raise: is the latest trade price a fair mark? what is a valid trade (positive quantity and price, side B or S)? are fees in bps of gross value? trades before a portfolio's inception date? no FX, so cross-currency totals are nominal.
- Terms to settle: AUM, position, latest price, gross value, fee, basis point, advisor book, valid trade.

</details>

<a id="s2"></a>

## 2. Phase 1: the baseline

### Challenge — Atlas Phase 1

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🏆 CHALLENGE TIME &nbsp;—&nbsp; Atlas capstone, Phase 1 &nbsp;·&nbsp; two to three hours</div>
<div style="background:#f8f8f8; padding:12px 14px; color:#1a1a1a;">Every team establishes the baseline: seeds, the three layers, the tests, and the orchestration understood. &nbsp;·&nbsp; <code>challenges/v1/04-atlas-capstone/README.md</code></div></div>

**Readiness, from the repo root**

1. `test -f .mise.toml && test -d data_products/atlas_elt && echo OK`.
2. `mise exec -- databricks auth describe --profile DEFAULT`.
3. `cd data_products/atlas_elt && dbt debug`.

**Sections**

1. 1 Inspect the source data.
2. 2 Understand the dbt project: `dbt deps`, `dbt ls`.
3. 3 Set up the Databricks connection.
4. 4 `dbt seed`.
5. 5 Build source-aligned: `dbt run --select tag:source-aligned`.
6. 6 Build domain-aligned.
7. 7 Build consumer-aligned and run the tests.
8. 8 Understand the orchestration.

**Gate.** `mise run challenge:verify 04` (not a bare verify.sh: it needs mise's environment); compare numbers with the expected results below.

**Support.**

- Edit only under `data_products/atlas_elt/`; the stubs under `challenges/` are reset templates; solutions live with the instructor.

### Circulating — section notes, expected results, blockers

| Section | What to check when circulating | Expected |
|---|---|---|
| 1 Seeds | Can they name the key of each seed and the join columns in trades? | trade_id; portfolio_id → portfolios; instrument_id → instruments |
| 2 Project | Do they see the tags per folder and that most models are views? | `dbt ls` lists the stub models; tags source-, domain-, consumer-aligned |
| 4 Seed | Row counts | trades 10,000; instruments 1,502; portfolios 2,501; clients 2,002; advisors 120; funds 60; fund_holdings 3,205 |
| 5 Source-aligned | Pass-through only; trades as a table | 7 models, same counts as the seeds |
| 6 Domain-aligned | Valid-trade filter; epoch to date; cents and bps; latest price by (ts, trade_id); positions net signed quantity | dom_trading__trade below 10,000; latest price 788 instruments |
| 7 Consumer-aligned and tests | The four reconciliations pass; one value checked by hand | See the expected results below |
| 8 Orchestration | Can they say what the DAG YAML triggers and where dbt actually runs? | A Databricks job per dataset, triggered by Airflow |

### Expected results (from the solution built on this laptop)
Numbers come from the supplied seeds and the solution models, so a learner's Databricks build of the same models should match. Money is in native trade currency (no FX), as the brief says.

## Row counts per model

| Model | Layer | Rows |
|---|---|---:|
| `agg_fund_nav` | consumer-aligned | 60 |
| `agg_instrument_activity` | consumer-aligned | 788 |
| `agg_segment_aum` | consumer-aligned | 4 |
| `api_latest_price` | consumer-aligned | 788 |
| `dom_fund__fund` | domain-aligned | 60 |
| `dom_fund__holding` | domain-aligned | 3,205 |
| `dom_instrument__instrument` | domain-aligned | 1,500 |
| `dom_party__advisor` | domain-aligned | 120 |
| `dom_party__client` | domain-aligned | 2,000 |
| `dom_portfolio__portfolio` | domain-aligned | 2,500 |
| `dom_positions__position` | domain-aligned | 3,717 |
| `dom_trading__latest_price` | domain-aligned | 788 |
| `dom_trading__trade` | domain-aligned | 9,880 |
| `portfolios_snapshot` | snapshot (Phase 2) | 2,500 |
| `rpt_advisor_book` | consumer-aligned | 120 |
| `rpt_advisor_leaderboard` | consumer-aligned | 120 |
| `rpt_portfolio_performance` | consumer-aligned | 1,028 |
| `rpt_portfolio_summary` | consumer-aligned | 1,028 |
| `rpt_region_report` | consumer-aligned | 6 |
| `src_crm__advisors` | source-aligned | 120 |
| `src_crm__clients` | source-aligned | 2,002 |
| `src_fundref__fund_holdings` | source-aligned | 3,205 |
| `src_fundref__funds` | source-aligned | 60 |
| `src_portfolio__portfolios` | source-aligned | 2,501 |
| `src_secmaster__instruments` | source-aligned | 1,502 |
| `src_trading__trades` | source-aligned | 10,000 |
| `ts_daily_asset_class` | consumer-aligned | 1,789 |
| `ts_daily_market` | consumer-aligned | 520 |
| `ts_daily_market_incremental` | consumer-aligned | 520 |

## Seeds

| seed | rows |
|---|---|
| trades | 10000 |
| instruments | 1502 |
| portfolios | 2501 |
| clients | 2002 |
| advisors | 120 |
| funds | 60 |
| fund_holdings | 3205 |

## Valid trades vs raw trades (the domain filter: qty > 0, price_cents > 0, side in B/S)

| raw_trades | valid_trades | trade_days | first_day | last_day |
|---|---|---|---|---|
| 10000 | 9880 | 520 | 2024-06-03 | 2026-05-29 |

## Advisor leaderboard, top 10 by book AUM

| rank | advisor_id | advisor_name | region | book_aum | fees_generated |
|---|---|---|---|---|---|
| 1 | 72 | William Vaughan | Zurich | 551,154,146.77 | 248,440.62 |
| 2 | 9 | Oliver Thorne | NewYork | 513,369,922.83 | 252,885.96 |
| 3 | 67 | Grace Sinclair | HongKong | 376,394,756.52 | 139,701.75 |
| 4 | 83 | Beatrice Hollis | NewYork | 372,009,975.21 | 190,797.90 |
| 5 | 39 | George Wexford | NewYork | 308,068,144.96 | 194,520.75 |
| 6 | 97 | Olivia Ashworth | Zurich | 289,219,472.32 | 182,172.14 |
| 7 | 27 | Edward Merrick | Zurich | 283,334,766.96 | 146,043.18 |
| 8 | 66 | Beatrice Merrick | Singapore | 280,294,530.73 | 120,631.84 |
| 9 | 54 | Edward Hollis | Zurich | 263,422,577.51 | 136,192.40 |
| 10 | 47 | Emily Thorne | NewYork | 263,086,127.41 | 155,160.38 |

## Segment AUM

| segment | clients | portfolios | aum |
|---|---|---|---|
| Institutional | 59 | 103 | 4,736,807,535.72 |
| UHNW | 119 | 192 | 1,697,924,446.95 |
| HNW | 200 | 227 | 236,390,671.74 |
| Retail | 506 | 506 | 50,695,494.58 |

## Region report

| region | portfolios | clients | aum | fees_generated |
|---|---|---|---|---|
| Zurich | 287 | 246 | 1,978,757,922.56 | 1,242,520.18 |
| NewYork | 241 | 197 | 1,693,714,454.67 | 1,050,718.43 |
| London | 207 | 185 | 1,083,797,948.90 | 719,318.14 |
| HongKong | 140 | 125 | 987,836,625.34 | 538,619.27 |
| Singapore | 85 | 73 | 694,630,664.49 | 365,299.46 |
| Dubai | 68 | 58 | 283,080,533.03 | 215,457.40 |

## Portfolio summary, five largest

| portfolio_id | strategy | client_name | segment | advisor_name | positions | aum | fees_paid | trades |
|---|---|---|---|---|---|---|---|---|
| 892 | Balanced | Tindall Investment Trust | Institutional | Olivia Sinclair | 12 | 133,208,598.49 | 41,453.88 | 29 |
| 2402 | Growth | Stonebridge Endowment | Institutional | Beatrice Merrick | 12 | 119,389,355.81 | 38,993.21 | 40 |
| 2377 | Income | Quillon Insurance Group | Institutional | William Vaughan | 9 | 116,662,315.59 | 37,899.94 | 30 |
| 2439 | Growth | Ravenswood Endowment | Institutional | Edward Merrick | 10 | 116,313,025.97 | 56,675.44 | 30 |
| 2026 | Balanced | Ravenswood Endowment | Institutional | William Vaughan | 12 | 92,022,957.40 | 30,716.16 | 31 |

## Totals that must reconcile (the four singular tests)

| summary_aum | advisor_book_aum | segment_aum | region_aum | position_trades | daily_trades | instrument_trades |
|---|---|---|---|---|---|---|
| 6,721,818,148.99 | 6,721,818,148.99 | 6,721,818,148.99 | 6,721,818,148.99 | 9880 | 9880 | 9880 |

## Instrument activity, top 5 by value traded

| ticker | asset_class | sector | qty_traded | value_traded | trades |
|---|---|---|---|---|---|
| JP8BJZVTNF85 | Bond | Government | 6880336 | 712,706,756.60 | 564 |
| HKC3770LK5N7 | Bond | Corporate | 7951573 | 703,106,665.08 | 480 |
| GBVPHF2MR9P4 | Bond | Corporate | 5555993 | 523,914,074.60 | 481 |
| 1730.HK | ETF | Broad | 4115442 | 521,185,213.86 | 612 |
| XZB | Equity | Industrials | 15175130 | 520,482,089.13 | 472 |

## Daily market, first 5 days

| trade_date | traded_value | fees | trades |
|---|---|---|---|
| 2024-06-03 | 3,288,376.48 | 2,410.15 | 11 |
| 2024-06-04 | 21,914,793.76 | 11,435.29 | 16 |
| 2024-06-05 | 11,305,809.08 | 6,394.09 | 14 |
| 2024-06-06 | 14,771,454.64 | 9,511.63 | 15 |
| 2024-06-07 | 8,659,413.97 | 4,294.40 | 10 |

## Fund NAV, top 5

| fund_id | name | style | holdings | nav_per_100_units |
|---|---|---|---|---|
| 4 | Orion Core Bond Fund | FixedIncome | 24 | 101.54 |
| 14 | Vantage Core Bond Fund | FixedIncome | 50 | 100.12 |
| 40 | Beacon Core Bond Fund | FixedIncome | 31 | 99.99 |
| 13 | Vantage Core Bond Fund | FixedIncome | 38 | 99.72 |
| 60 | Orion Core Bond Fund | FixedIncome | 27 | 98.10 |

## Latest price lookup, sample

| instrument_id | last_price | last_ts |
|---|---|---|
| 1 | 53.80 | 1780068649 |
| 2 | 109.95 | 1780049110 |
| 3 | 132.02 | 1780071034 |
| 4 | 41.10 | 1780049026 |
| 5 | 5.71 | 1780053433 |

In [ ]:
sql("select rank, advisor_name, region, round(book_aum, 2) as book_aum from training_demo.rpt_advisor_leaderboard order by rank limit 5", HERE / "support/atlas_duckdb/warehouse/training_dev.duckdb")
# Compare a team's advisor book with this; a different top five usually means a different latest-price rule or an unfiltered trade set.

In [ ]:
sql("""select (select count(*) from training_demo.src_trading__trades) as raw_trades, (select count(*) from training_demo.dom_trading__trade) as valid_trades,
                 round((select sum(aum) from training_demo.rpt_portfolio_summary), 2) as summary_aum, round((select sum(book_aum) from training_demo.rpt_advisor_book), 2) as advisor_book_aum""", HERE / "support/atlas_duckdb/warehouse/training_dev.duckdb")

| Symptom | Cause | Fix |
|---|---|---|
| `tag:source-aligned` selects nothing | The `models:` block in `dbt_project.yml` was edited | Restore the folder-level tags; they are per layer folder, not in model bodies |
| `dbt deps` fails or `ref` to a shared macro is missing | The local shared package was not installed | `dbt deps` from `data_products/atlas_elt` |
| A bare `bash verify.sh` fails on auth | It runs outside mise's environment | `mise run challenge:verify 04` |
| `show --inline "select …"` errors | mise word-splits the SQL | `dbt show --select <model>` |
| `bundle validate` warns about an expired key | A sandbox clock artefact | Non-fatal; the dbt gate still runs |
| Target path under `/Volumes/...` is not mounted | The project writes artefacts to a Unity Catalog volume in production | Add `--target-path target` for local inspection and clean it up after |
| Transient "Error during request to server" | Databricks | Retry |

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🍽️ LUNCH &nbsp;·&nbsp; 60 min &nbsp;·&nbsp; back at 13:30 at the latest</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

<a id="s3"></a>

## 3. Phase 2: extensions and presentation prep

### Challenge — choose extensions by value to the consumer; technical work stops at 14:30

Choose by value to the consumer, not by difficulty. Each is a README challenge with a solution kept by the instructor; the DuckDB build in `support/atlas_duckdb/` has them all, for comparing a team's result.

| Group | Extensions |
|---|---|
| Richer consumer marts | 1 executive report mart · 2 advisor leaderboard with `dense_rank()` · 3 daily activity by asset class with a rolling window · 14 a reconciling regional report (the Phase 2 capstone) |
| Documentation and freshness | 4 source freshness on the extracts · 5 exposures: who consumes the marts |
| Reuse and tests | 6 a reusable money macro · 7 a custom generic test for non-negative money · 8 a relationship test · 9 singular reconciliation tests within tolerance |
| Loading patterns | 10 an incremental market mart · 11 a snapshot for slowly changing portfolios |
| Orchestration | 12 understand and extend the orchestration · 13 params-driven orchestration |

In [ ]:
!cd support/atlas_duckdb && dbt ls --resource-type model --select tag:consumer-aligned && dbt ls --resource-type test | head -12
# The solution's consumer-aligned models and tests, for reference while a team describes theirs.

<a id="s4"></a>

## 4. Presentations and close

### Activity 13 — capstone presentation

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 13 &nbsp;—&nbsp; Capstone presentation &nbsp;·&nbsp; One person per table &nbsp;·&nbsp; 3 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Two to three minutes: what you built and for whom, what was hard and what was easy, one finding in the data, one trust question you answered and how, and what you would do next. If you used Catalyst or the design documents, show one. If senior leaders are in the room, say where the paved path and Catalyst helped.</strong><br><br><strong>Produce:</strong><br>• The outputs you built, with one number you checked independently<br>• The hardest step, and the finding<br>• One trust question and its answer<br><br><strong>Debrief questions:</strong><br>• Which teams picked the same outputs, and did their numbers agree?<br>• What would you tell the next cohort before they start?</div></div>

### Tally sheet and certification evidence

**Tally sheet** (fill in during the day)

| Table | Outputs chosen | Phase 1 built (src / dom / consumer) | Tests pass | Independent check shown | Orchestration explained | Phase 2 taken | Presenter |
|---|---|---|---|---|---|---|---|
| 1 |  |  |  |  |  |  |  |
| 2 |  |  |  |  |  |  |  |
| 3 |  |  |  |  |  |  |  |
| 4 |  |  |  |  |  |  |  |
| 5 |  |  |  |  |  |  |  |

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 CERTIFICATION EVIDENCE — what Phase 1 shows per team</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Observed during the day, not handed in.</div></div>

- The worksheet: consumer, decision, sources, rules, assumptions.
- The three layers built with the tag selectors.
- The tests passing, including the reconciliations.
- One value checked independently of the model that produced it.
- An explanation of what the orchestration triggers and where dbt runs.
- A presentation that states a finding and a caveat.

### Close

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">✅ BEFORE LEARNERS LEAVE</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">☐ Each team's Phase 1 evidence recorded on the tally sheet: layers built, tests passed, one independent value check, orchestration explained.<br>☐ Each team's worksheet and presentation kept with their project.<br>☐ The week's artefacts (PRDs, contracts, STTMs, generated code, dbt and Airflow projects) saved where the learners can find them.<br>☐ The list of open questions for the platform team handed over with names.</div></div>

<a id="questions"></a>

## Questions from the room

Type the question in the table as it comes; answer live in the cells below; tidy up at the break. Anything unanswered goes to the platform team or the Teams channel, with the person who asked.

| # | Question | Who asked | Answer / where it went |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

In [ ]:
# Answer live here: SQL against a demo warehouse, a small Python check, or a shell command.
# Example: sql("select * from training_demo.stg_holdings limit 5", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")

In [ ]:
# Sketch a diagram live (before Excalidraw). Edit and run.
quick_diagram("""
flowchart LR
  A[source file] --> B[dbt rules on Databricks] --> C[consumer table]
""")

**Answers written up after the day** — fill in here, then share the notebook.

- 

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>